In [1]:
#"Yolo modelini kullanabilmek için kutuphane kurulumları"
!pip install ultralytics

In [2]:
import os

path = '/kaggle/input/datasets'
print(os.listdir(path))

for item in os.listdir(path):
    full = os.path.join(path, item)
    if os.path.isdir(full):
        print(f"\n{item}/", os.listdir(full)[:10])

['zlemdemir', 'tudorhirtopanu']

zlemdemir/ ['operator-golgeleme-video']

tudorhirtopanu/ ['yolo-highvis-and-person-detection-dataset']


In [3]:
path = '/kaggle/input/datasets/tudorhirtopanu/yolo-highvis-and-person-detection-dataset'
print(os.listdir(path))
path = '/kaggle/input/datasets/tudorhirtopanu/yolo-highvis-and-person-detection-dataset/YOLO-HiVis-Data'
print(os.listdir(path))

['YOLO-HiVis-Data']
['labels', 'images']


In [4]:
import os, shutil, random

dataset_path = '/kaggle/input/datasets/tudorhirtopanu/yolo-highvis-and-person-detection-dataset/YOLO-HiVis-Data'
output_path = '/kaggle/working/dataset'
# eğitimde kullanılacak train val ve test klasörleri oluşturuldu
for split in ['train', 'val', 'test']:
    os.makedirs(f'{output_path}/{split}/images', exist_ok=True)
    os.makedirs(f'{output_path}/{split}/labels', exist_ok=True)

images = [f for f in os.listdir(f'{dataset_path}/images') if f.endswith('.jpg')]
random.seed(42)
random.shuffle(images)

# veri setinin %70 i eğitim, %20 si validasyon %10 u da test olarak ayırdık.
train_end = int(len(images) * 0.7)
val_end = int(len(images) * 0.9)

splits = {
    'train': images[:train_end],
    'val': images[train_end:val_end],
    'test': images[val_end:]
}

for split, file_list in splits.items():
    for img in file_list:
        label = img.replace('.jpg', '.txt')
        shutil.copy(f'{dataset_path}/images/{img}', f'{output_path}/{split}/images/{img}')
        shutil.copy(f'{dataset_path}/labels/{label}', f'{output_path}/{split}/labels/{label}')

for split in ['train', 'val', 'test']:
    count = len(os.listdir(f'{output_path}/{split}/images'))
    print(f'{split}: {count} görüntü')

train: 5555 görüntü
val: 1588 görüntü
test: 794 görüntü


In [5]:
yaml_content = """
train: /kaggle/working/dataset/train/images
val: /kaggle/working/dataset/val/images
test: /kaggle/working/dataset/test/images

nc: 2
names: ['person', 'high-vis']
"""

with open('/kaggle/working/dataset/data.yaml', 'w') as f:
    f.write(yaml_content)

print("data.yaml oluştu")

data.yaml oluştu


In [6]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data='/kaggle/working/dataset/data.yaml',
    epochs=20,
    imgsz=1280,
    batch=8,
    name='hivis_deneme_2',
    mosaic=1.0,       
    mixup=0.2,        
    scale=0.5,        
    fliplr=0.5,
    patience=10,
)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=hivis_deneme_2-4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tru

In [13]:
from ultralytics import YOLO

model = YOLO('/kaggle/working/runs/detect/hivis_deneme_2-4/weights/best.pt')

print(model.info())

results = model.val(data='/kaggle/working/dataset/data.yaml', split='test')

Model summary: 130 layers, 11,136,374 parameters, 0 gradients, 28.6 GFLOPs
(130, 11136374, 0, 28.6491136)
Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 73 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 803.4±361.0 MB/s, size: 37.4 KB)
val: Scanning /kaggle/working/dataset/test/labels.cache... 794 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 794/794 277.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 50/50 1.4it/s 36.8s0.8ss
                   all        794       2580      0.857      0.831      0.894      0.639
                person        546       1283      0.835      0.769      0.847      0.591
              high-vis        739       1297      0.879      0.894       0.94      0.686
Speed: 3.5ms preprocess, 38.6ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /kaggle

In [14]:
import shutil
shutil.copy(
    '/kaggle/working/runs/detect/hivis_deneme_2-4/weights/best.pt',
    '/kaggle/working/best_v2.pt'
)
print("Model kopyalandı, Output'tan indirebilirsin")

Model kopyalandı, Output'tan indirebilirsin
